In [1]:
import os
print("Thư mục hiện tại của Notebook:", os.getcwd())
print("Danh sách file/thư mục ở thư mục hiện tại:", os.listdir("."))

Thư mục hiện tại của Notebook: e:\cơ sở hệ thông thông tin\Co-design-of-Human-centered-Explainable-AI-for-Clinical-Decision-Support\Notebook
Danh sách file/thư mục ở thư mục hiện tại: ['01_data_preprocessing.ipynb', '02_train_doctor_ai.ipynb', '03_doctor_xai_engine.ipynb', '04_technical_validation.ipynb']


In [2]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path(r"E:\cơ sở hệ thông thông tin\Co-design-of-Human-centered-Explainable-AI-for-Clinical-Decision-Support")

found_files = list(project_root.glob("**/admissions.csv*"))

if found_files:
    # Lấy thư mục chứa file admissions.csv vừa tìm thấy
    DATA_RAW_DIR = str(found_files[0].parent)
    print("ĐÃ TÌM THẤY THƯ MỤC CHỨA DỮ LIỆU CHÍNH XÁC:")
    print(DATA_RAW_DIR)
    print("\nDanh sách các file thực tế có trong thư mục:")
    print(os.listdir(DATA_RAW_DIR))
else:
    print("Chưa tìm thấy file 'admissions.csv' trong dự án!")
    print("Em hãy kiểm tra xem file có đang nằm trong file nén (.zip / .tar.gz) chưa giải nén không nhé.")

DATA_PROCESSED_DIR = os.path.join(str(project_root), "data", "processed")
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)



ĐÃ TÌM THẤY THƯ MỤC CHỨA DỮ LIỆU CHÍNH XÁC:
E:\cơ sở hệ thông thông tin\Co-design-of-Human-centered-Explainable-AI-for-Clinical-Decision-Support\mimic-iv-clinical-database-demo-2.2\hosp

Danh sách các file thực tế có trong thư mục:
['admissions.csv.gz', 'diagnoses_icd.csv.gz', 'drgcodes.csv.gz', 'd_hcpcs.csv.gz', 'd_icd_diagnoses.csv.gz', 'd_icd_procedures.csv.gz', 'd_labitems.csv.gz', 'emar.csv.gz', 'emar_detail.csv.gz', 'hcpcsevents.csv.gz', 'labevents.csv.gz', 'microbiologyevents.csv.gz', 'omr.csv.gz', 'patients.csv.gz', 'pharmacy.csv.gz', 'poe.csv.gz', 'poe_detail.csv.gz', 'prescriptions.csv.gz', 'procedures_icd.csv.gz', 'provider.csv.gz', 'services.csv.gz', 'transfers.csv.gz']


In [3]:

df_adm = pd.read_csv(
    os.path.join(DATA_RAW_DIR, "admissions.csv.gz"),
    usecols=["subject_id", "hadm_id", "admittime"]
)

df_diag = pd.read_csv(
    os.path.join(DATA_RAW_DIR, "diagnoses_icd.csv.gz"),
    usecols=["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"]
)

df_adm["admittime"] = pd.to_datetime(df_adm["admittime"])

print(f"ĐÃ ĐỌC THÀNH CÔNG DỮ LIỆU!")
print(f"Tổng số lượt nhập viện (admissions): {len(df_adm)}")
print(f"Tổng số hồ sơ chẩn đoán (diagnoses): {len(df_diag)}")

ĐÃ ĐỌC THÀNH CÔNG DỮ LIỆU!
Tổng số lượt nhập viện (admissions): 275
Tổng số hồ sơ chẩn đoán (diagnoses): 4506


In [4]:
df_diag_icd9 = df_diag[df_diag["icd_version"] == 9].copy()

df_diag_icd9.dropna(subset=["icd_code"], inplace=True)
df_diag_icd9["icd_code"] = df_diag_icd9["icd_code"].astype(str).str.strip()

visit_counts = df_diag_icd9.groupby("subject_id")["hadm_id"].nunique()

valid_patients = visit_counts[visit_counts >= 2].index.tolist()

print(f"Số lượng bệnh nhân có >= 2 lượt khám (ICD-9): {len(valid_patients)}")

df_filtered_diag = df_diag_icd9[df_diag_icd9["subject_id"].isin(valid_patients)]

Số lượng bệnh nhân có >= 2 lượt khám (ICD-9): 28


In [5]:

df_merged = pd.merge(
    df_filtered_diag, 
    df_adm[["hadm_id", "admittime"]], 
    on="hadm_id", 
    how="inner"
)

df_merged.sort_values(
    by=["subject_id", "admittime", "seq_num"], 
    ascending=[True, True, True], 
    inplace=True
)

print("Ví dụ 5 dòng dữ liệu sau khi sắp xếp:")
display(df_merged.head())

Ví dụ 5 dòng dữ liệu sau khi sắp xếp:


,subject_id,hadm_id,seq_num,icd_code,icd_version,admittime
617,10000032,22595853,1,5723,9,2180-05-06 22:23:00
619,10000032,22595853,2,78959,9,2180-05-06 22:23:00
616,10000032,22595853,3,5715,9,2180-05-06 22:23:00
620,10000032,22595853,4,07070,9,2180-05-06 22:23:00
615,10000032,22595853,5,496,9,2180-05-06 22:23:00


In [6]:
def icd9_to_ccs_category(icd_code):
    
    clean_code = icd_code.replace(".", "").upper()
    
    prefix = clean_code[:3]
    
    if prefix.isdigit():
        num = int(prefix)
        if 390 <= num <= 459:
            return "CCS_CIRCULATORY"  # Bệnh hệ tuần hoàn (Tim mạch, Huyết áp...)
        elif 460 <= num <= 519:
            return "CCS_RESPIRATORY"  # Bệnh hệ hô hấp
        elif 520 <= num <= 579:
            return "CCS_DIGESTIVE"    # Bệnh hệ tiêu hóa
        elif 240 <= num <= 279:
            return "CCS_ENDOCRINE"    # Bệnh nội tiết, chuyển hóa [8]
        elif 580 <= num <= 629:
            return "CCS_GENITOURINARY" # Bệnh tiết niệu/sinh dục
    
    return "CCS_OTHER"

df_merged["ccs_code"] = df_merged["icd_code"].apply(icd9_to_ccs_category)

In [7]:
patient_sequences = {}

for subject_id, group_patient in df_merged.groupby("subject_id", sort=False):
    patient_visits = []
    
    for hadm_id, group_visit in group_patient.groupby("hadm_id", sort=False):
        icd_list = group_visit["icd_code"].unique().tolist()
        patient_visits.append(icd_list)
    
    patient_sequences[str(subject_id)] = patient_visits

patient_keys = list(patient_sequences.keys())
if len(patient_keys) > 0:
    sample_patient_id = patient_keys[0]
print(f"Cấu trúc chuỗi khám của Bệnh nhân ID {sample_patient_id}:")
print(json.dumps(patient_sequences[sample_patient_id], indent=2))

Cấu trúc chuỗi khám của Bệnh nhân ID 10000032:
[
  [
    "5723",
    "78959",
    "5715",
    "07070",
    "496",
    "29680",
    "30981",
    "V1582"
  ],
  [
    "07071",
    "78959",
    "2875",
    "2761",
    "496",
    "5715",
    "V08",
    "3051"
  ],
  [
    "45829",
    "07044",
    "7994",
    "2761",
    "78959",
    "2767",
    "3051",
    "V08",
    "V4986",
    "V462",
    "496",
    "29680",
    "5715"
  ],
  [
    "07054",
    "78959",
    "V462",
    "5715",
    "2767",
    "2761",
    "496",
    "V08",
    "3051",
    "78791"
  ]
]


In [10]:
from sklearn.model_selection import train_test_split

patient_ids = list(patient_sequences.keys())

# Chia tập Train (80%) và Test (20%)
train_ids, test_ids = train_test_split(patient_ids, test_size=0.20, random_state=42)

# Chia tiếp tập Train thành Train và Validation (15% cho Val)
train_ids, val_ids = train_test_split(train_ids, test_size=0.15, random_state=42)
print(f"Số bệnh nhân tập Train: {len(train_ids)}")
print(f"Số bệnh nhân tập Validation: {len(val_ids)}")
print(f"Số bệnh nhân tập Test: {len(test_ids)}")

processed_data = {
    "sequences": patient_sequences,
    "splits": {
        "train": train_ids,
        "val": val_ids,
        "test": test_ids
    }
}

output_filepath = os.path.join(DATA_PROCESSED_DIR, "patient_sequences.json")
with open(output_filepath, "w") as f:
    json.dump(processed_data, f, indent=2)

print(f"\nĐã lưu thành công dữ liệu tiền xử lý vào: {output_filepath}")

Số bệnh nhân tập Train: 18
Số bệnh nhân tập Validation: 4
Số bệnh nhân tập Test: 6

Đã lưu thành công dữ liệu tiền xử lý vào: E:\cơ sở hệ thông thông tin\Co-design-of-Human-centered-Explainable-AI-for-Clinical-Decision-Support\data\processed\patient_sequences.json
